# 

In [1]:
import torch
print(torch.cuda.is_available())
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
import numpy as np
import torch.optim as optim
import sys



True


In [2]:
from fast_dataset_open import open_with_cache
# from cache_models import (
#     train_with_cache, load_model_with_cache, evaluate_with_cache
# )
# from utils_ import plot_all_roc_curves, load_models_with_cache, train_models_with_cache, evaluate_models_with_cache
# from models_params import models_params

In [3]:
dataset_dir = "datasets/TEP/"

In [4]:

training_fault_free_df = open_with_cache(f"{dataset_dir}/TEP_FaultFree_Training.RData")
validation_fault_free_df = open_with_cache(f"{dataset_dir}/TEP_FaultFree_Testing.RData")

print(f"training dataset size GB: {sys.getsizeof(training_fault_free_df)/1e9}")
print(f"validation dataset size GB: {sys.getsizeof(validation_fault_free_df)/1e9}")

Loading from cache: cache_data/40501dd3f9438f820d63e297020b0214.pkl
Loading from cache: cache_data/205a0885a9383d02f60aca68cc1d8fec.pkl
training dataset size GB: 0.109000144
validation dataset size GB: 0.207360144


In [5]:
X_columns = ['simulationRun',  'xmeas_1', 'xmeas_2', 'xmeas_3', 'xmeas_4', 'xmeas_5', 'xmeas_6', 'xmeas_7', 'xmeas_8', 'xmeas_9', 'xmeas_10', 'xmeas_11', 'xmeas_12', 'xmeas_13', 'xmeas_14', 'xmeas_15', 'xmeas_16', 'xmeas_17', 'xmeas_18', 'xmeas_19', 'xmeas_20', 'xmeas_21', 'xmeas_22', 'xmeas_23', 'xmeas_24', 'xmeas_25', 'xmeas_26', 'xmeas_27', 'xmeas_28', 'xmeas_29', 'xmeas_30', 'xmeas_31', 'xmeas_32', 'xmeas_33', 'xmeas_34', 'xmeas_35', 'xmeas_36', 'xmeas_37', 'xmeas_38', 'xmeas_39', 'xmeas_40', 'xmeas_41', 'xmv_1', 'xmv_2', 'xmv_3', 'xmv_4', 'xmv_5', 'xmv_6', 'xmv_7', 'xmv_8', 'xmv_9', 'xmv_10', 'xmv_11']

In [6]:
X_train = training_fault_free_df[X_columns].values
X_val = validation_fault_free_df[X_columns].values

In [7]:
X_train_norm = X_train.copy()
X_val_norm = X_val.copy()


In [8]:
X_means = X_train_norm.mean(axis=0)[1:]
X_stds = X_train_norm.std(axis=0)[1:]
X_train_norm[:,1:] = (X_train_norm[:,1:]-X_means)/X_stds
X_val_norm[:,1:] = (X_val_norm[:,1:]-X_means)/X_stds

In [9]:
train_samples_per_run = np.count_nonzero(X_train[:,0] ==1)
val_samples_per_run = np.count_nonzero(X_val[:,0] ==1)
train_samples_per_run, val_samples_per_run

(500, 960)

In [54]:

batch_size = 2048
seq_len = 120//3 # 2 hours
input_dim = 52

In [11]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [12]:

class TEPDatasetEfficient(Dataset):
    def __init__(self, data, seq_len=seq_len, samples_per_run = 500):
        self.data = data
        self.seq_len = seq_len
        self.feature_cols = slice(1, data.shape[1])

        # Index simulation boundaries
        self.sim_ids, sim_start_idxs = np.unique(data[:, 0], return_index=True)
        self.sim_start_idxs = sim_start_idxs
        self.num_sims = len(self.sim_ids)
        self.samples_per_sim = samples_per_run- seq_len

        # Total samples
        self.total_samples = self.num_sims * self.samples_per_sim

    def __len__(self):
        return self.total_samples

    def __getitem__(self, idx):
        # Determine which simulation this index belongs to
        sim_idx = idx // self.samples_per_sim
        seq_start_idx = idx % self.samples_per_sim

        # Calculate the global index in the original numpy array
        global_start_idx = self.sim_start_idxs[sim_idx] + seq_start_idx

        # Extract sequence and target without copying (views only)
        seq = self.data[global_start_idx:global_start_idx + self.seq_len, self.feature_cols]
        target = self.data[global_start_idx + self.seq_len, self.feature_cols]

        # Convert to torch tensors (minimal memory overhead)
        seq_tensor = torch.from_numpy(seq).float()
        target_tensor = torch.from_numpy(target).float()

        return seq_tensor, target_tensor


class TEPDatasetGPU(Dataset):
    def __init__(self, features_gpu, sim_ids, sim_start_idxs, seq_len=seq_len, samples_per_run = 500):
        self.features_gpu = features_gpu
        self.seq_len = seq_len
        self.sim_start_idxs = sim_start_idxs
        self.num_sims = len(sim_ids)
        self.samples_per_sim = samples_per_run - seq_len
        self.total_samples = self.num_sims * self.samples_per_sim

    def __len__(self):
        return self.total_samples

    def __getitem__(self, idx):
        sim_idx = idx // self.samples_per_sim
        seq_start_idx = idx % self.samples_per_sim
        global_start_idx = self.sim_start_idxs[sim_idx] + seq_start_idx

        seq = self.features_gpu[global_start_idx:global_start_idx + self.seq_len]  # GPU views
        target = self.features_gpu[global_start_idx + self.seq_len]

        return seq, target



In [13]:
# tep_train_dataset = TEPDatasetEfficient(X_train_norm , seq_len=seq_len, samples_per_run=train_samples_per_run)
# tep_val_dataset = TEPDatasetEfficient(X_val_norm , seq_len=seq_len, samples_per_run=val_samples_per_run)
train_sim_ids, train_sim_start_idxs = np.unique(X_train_norm[:, 0], return_index=True)
val_sim_ids, val_sim_start_idxs = np.unique(X_val_norm[:, 0], return_index=True)

X_train_norm_gpu = torch.tensor(X_train_norm[:,1:], dtype=torch.float32, device='cuda')
X_val_norm_gpu = torch.tensor(X_val_norm[:,1:], dtype=torch.float32, device='cuda')

tep_train_dataset = TEPDatasetGPU(X_train_norm_gpu ,train_sim_ids, train_sim_start_idxs, seq_len=seq_len, samples_per_run=train_samples_per_run)
tep_val_dataset = TEPDatasetGPU(X_val_norm_gpu, val_sim_ids,val_sim_start_idxs , seq_len=seq_len, samples_per_run=val_samples_per_run)

In [14]:
train_loader = DataLoader(tep_train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(tep_val_dataset, batch_size=batch_size, shuffle=True, num_workers=0)

In [67]:
class TEPTransformer(nn.Module):
    def __init__(self, input_dim=52, seq_len=10, d_model=32, nhead=32, num_layers=1, dim_feedforward=32):
        super().__init__()

        self.embedding = nn.Linear(input_dim, d_model)
        self.pos_encoder = nn.Parameter(torch.zeros(1, seq_len, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            activation="relu",
            batch_first=True
        )

        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.regressor = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.ReLU(),
            nn.Linear(dim_feedforward, input_dim)
        )

    def forward(self, x):
        # x: (batch_size, seq_len, 53)
        x = self.embedding(x) + self.pos_encoder  # add positional encoding
        transformer_out = self.transformer_encoder(x)  # (batch_size, seq_len, d_model)

        # Use the last output timestep to predict next state
        last_step_out = transformer_out[:, -1, :]  # (batch_size, d_model)

        predicted_next_state = self.regressor(last_step_out)  # (batch_size, 53)

        return predicted_next_state


In [62]:
import torch
import torch.nn as nn
import math

class TEPTransformerGPT(nn.Module):
    def __init__(self, input_dim=52, seq_len=10, d_model=52, nhead=52, num_layers=2, dim_feedforward=52):
        super().__init__()
        self.seq_len = seq_len
        self.d_model = d_model

        # Embedding input to model dimension
        self.embedding = nn.Linear(input_dim, d_model)

        # Sinusoidal positional encoding
        self.positional_encoding = self._generate_positional_encoding(seq_len, d_model)

        # Transformer encoder layer
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            activation="relu",
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Attention pooling (learned query vector)
        self.query_vector = nn.Parameter(torch.randn(d_model))

        # Regressor to predict next state
        self.regressor = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.ReLU(),
            nn.Linear(dim_feedforward, input_dim)
        )

    def _generate_positional_encoding(self, seq_len, d_model):
        pe = torch.zeros(seq_len, d_model)
        position = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe.unsqueeze(0)  # shape: (1, seq_len, d_model)

    def _generate_causal_mask(self, size):
        return torch.triu(torch.ones(size, size) * float('-inf'), diagonal=1)

    def forward(self, x):
        # x: (batch_size, seq_len, input_dim)
        batch_size = x.size(0)
        x = self.embedding(x) + self.positional_encoding.to(x.device)

        # Apply causal mask
        mask = self._generate_causal_mask(self.seq_len).to(x.device)
        transformer_out = self.transformer_encoder(x, mask=mask)

        # Attention pooling
        # Compute similarity between each time step and the learned query vector
        query = self.query_vector.unsqueeze(0).unsqueeze(0)  # (1, 1, d_model)
        attn_scores = torch.matmul(transformer_out, query.transpose(-2, -1)).squeeze(-1)  # (batch_size, seq_len)
        attn_weights = torch.softmax(attn_scores, dim=1).unsqueeze(-1)  # (batch_size, seq_len, 1)

        pooled = torch.sum(transformer_out * attn_weights, dim=1)  # (batch_size, d_model)
        predicted_next_state = self.regressor(pooled)  # (batch_size, input_dim)

        return predicted_next_state


In [68]:



model = TEPTransformer(input_dim=input_dim, seq_len=seq_len).to('cuda')
#model= TEPTransformerGPT(input_dim=input_dim, seq_len=seq_len).to('cuda')

# # Dummy input data
# dummy_input = torch.randn(batch_size, seq_len, input_dim)

# # Forward pass
# predicted_next_state = model(dummy_input)

# print(predicted_next_state.shape)  # should be (batch_size, 53)


In [22]:
criterion = torch.nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

num_epochs = 20

for epoch in range(num_epochs):
    model.train()
    train_loss= 0
    # trainig loop
    for seq, target in train_loader:
        #seq, target = seq.to('cuda'), target.to('cuda')

        optimizer.zero_grad()
        output = model(seq)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        train_loss+= loss.item() * seq.size(0)

    train_loss = train_loss/ len(train_loader.dataset)
    model.eval()
    val_loss = 0
    with torch.no_grad():
        # validation loop
        for seq, target in val_loader:
            #seq, target = seq.to('cuda'), target.to('cuda')
    
            output = model(seq)
            loss = criterion(output, target)
    
            val_loss += loss.item() * seq.size(0)
    
        val_loss= val_loss/ len(train_loader.dataset)
    print(f"Epoch {epoch+1}/{num_epochs}, Training Loss: {train_loss:.6f}, Training Loss: {val_loss:.6f}")

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu! (when checking argument for argument mat1 in method wrapper_CUDA_addmm)

In [11]:
# X_train_norm_output = X_train_norm[seq_len:]
# X_train_norm_output.shape
# #X_train_norm_input = X_train_norm[seq_len:]

In [12]:
# indexs = np.lib.stride_tricks.sliding_window_view(np.array(range(X_train_norm.shape[0])), seq_len)
# print(indexs.shape)
# indexs = indexs.flatten()
# ind_ = indexs.reshape(-1, seq_len)
# print(ind_.shape)

In [13]:
# X_train_norm_input = X_train_norm[indexs].reshape(-1,seq_len, 53)

In [14]:
# X_train_norm_input.shape

In [15]:
# X_train_norm_input[475,:,0]


In [13]:
def df_to_dataset(df):
    samples_per_run = np.count_nonzero(df["simulationRun"] ==1)
    indxs = np.lib.stride_tricks.sliding_window_view(np.array(range(samples_per_run-1)), seq_len)
    indxs = indxs.flatten()
    inputs_ = []
    for sim_run in np.unique(df["simulationRun"]):
        run_df = df[df["simulationRun"] == sim_run][X_columns[1:]]
        run_input = run_df.values[indxs].reshape(-1, seq_len, input_dim)
        inputs_.append(run_input)
        
    input_ = np.concat(inputs_)
    input_ = (input_-X_means)/X_stds
        
    output_ = df[df["sample"]>(seq_len)][X_columns[1:]].values
    output_ = (output_-X_means)/X_stds
    input_tnsr = torch.tensor(input_, dtype=torch.float32)
    output_tnsr = torch.tensor(output_, dtype=torch.float32)
    
    return TensorDataset(input_tnsr, output_tnsr)
    

In [15]:
val_dataset = df_to_dataset(validation_fault_free_df[validation_fault_free_df["simulationRun"]<50])

In [16]:
train_dataset = df_to_dataset(training_fault_free_df)


In [76]:
print(train_dataset.tensors[0].shape)

torch.Size([230000, 40, 52])


In [18]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True)

In [70]:
criterion = torch.nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 20

for epoch in range(num_epochs):
    model.train()
    train_loss= 0
    # trainig loop
    for seq, target in train_loader:
        seq, target = seq.to('cuda'), target.to('cuda')

        optimizer.zero_grad()
        output = model(seq)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        train_loss+= loss.item() * seq.size(0)

    train_loss = train_loss/ len(train_loader.dataset)
    model.eval()
    val_loss = 0
    with torch.no_grad():
        # validation loop
        for seq, target in val_loader:
            seq, target = seq.to('cuda'), target.to('cuda')
    
            output = model(seq)
            loss = criterion(output, target)
    
            val_loss += loss.item() * seq.size(0)
    
        val_loss= val_loss/ len(val_loader.dataset)
    print(f"Epoch {epoch+1}/{num_epochs}, Training Loss: {train_loss:.6f}, Validation Loss: {val_loss:.6f}")

Epoch 1/20, Training Loss: 0.705157, Validation Loss: 0.601629
Epoch 2/20, Training Loss: 0.578379, Validation Loss: 0.558007
Epoch 3/20, Training Loss: 0.554773, Validation Loss: 0.547201
Epoch 4/20, Training Loss: 0.546802, Validation Loss: 0.541896
Epoch 5/20, Training Loss: 0.542634, Validation Loss: 0.536388
Epoch 6/20, Training Loss: 0.525257, Validation Loss: 0.503385


KeyboardInterrupt: 

In [52]:
import torch
import torch.nn as nn
import math
import numpy as np

# ----------- ProbSparse Attention -----------
class ProbAttention(nn.Module):
    def __init__(self, mask_flag=False, factor=5, attention_dropout=0.1):
        super().__init__()
        self.factor = factor
        self.dropout = nn.Dropout(attention_dropout)
        self.mask_flag = mask_flag

    def forward(self, queries, keys, values, attn_mask=None):
        B, H, L_Q, D = queries.shape
        _, _, L_K, _ = keys.shape

        U_part = self.factor * int(np.ceil(np.log(L_K)))  # sparse factor
        scores_top = torch.einsum("bhqd,bhkd->bhqk", queries, keys) / math.sqrt(D)

        if self.mask_flag and attn_mask is not None:
            scores_top = scores_top.masked_fill(attn_mask, -np.inf)

        # Top-k sparsity
        index = scores_top.topk(U_part, dim=-1)[1]  # (B, H, L_Q, U_part)
        scores = torch.gather(scores_top, -1, index)
        attn = self.dropout(torch.softmax(scores, dim=-1))
        V_reduced = torch.gather(values.unsqueeze(2).expand(-1, -1, L_Q, -1, -1), 3, index.unsqueeze(-1).expand(-1, -1, -1, -1, values.shape[-1]))
        context = torch.sum(attn.unsqueeze(-1) * V_reduced, dim=-2)  # (B, H, L_Q, D)

        return context

# ----------- Attention Layer Wrapper -----------
class AttentionLayer(nn.Module):
    def __init__(self, attention, d_model, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.attention = attention

    def forward(self, x):
        B, L, D = x.shape
        H = self.n_heads

        q = self.q_proj(x).view(B, L, H, self.d_k).transpose(1, 2)
        k = self.k_proj(x).view(B, L, H, self.d_k).transpose(1, 2)
        v = self.v_proj(x).view(B, L, H, self.d_k).transpose(1, 2)

        context = self.attention(q, k, v)  # (B, H, L, D)
        context = context.transpose(1, 2).reshape(B, L, D)
        return self.out_proj(context)

# ----------- Encoder Layer -----------
class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn = AttentionLayer(ProbAttention(), d_model, n_heads)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x2 = self.attn(x)
        x = self.norm1(x + self.dropout(x2))
        x2 = self.ff(x)
        x = self.norm2(x + self.dropout(x2))
        return x

# ----------- Positional Encoding -----------
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(pos * div_term)
        pe[:, 1::2] = torch.cos(pos * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

# ----------- Lightweight Informer Model -----------
class MiniInformer(nn.Module):
    def __init__(self, input_dim, seq_len, d_model=64, n_heads=4, e_layers=2, d_ff=128, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_enc = PositionalEncoding(d_model)
        self.encoder = nn.Sequential(*[
            EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(e_layers)
        ])
        self.output_layer = nn.Linear(d_model, input_dim)  # predict next state

    def forward(self, x):
        """
        x: (batch_size, seq_len, input_dim)
        Returns: (batch_size, input_dim) - the next predicted state
        """
        x = self.input_proj(x)
        x = self.pos_enc(x)
        x = self.encoder(x)
        return self.output_layer(x[:, -1, :])  # predict based on the last token



In [58]:


model = MiniInformer(input_dim=input_dim, seq_len=seq_len).to("cuda")

In [59]:

criterion = torch.nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 20

for epoch in range(num_epochs):
    model.train()
    train_loss= 0
    # trainig loop
    for seq, target in train_loader:
        seq, target = seq.to('cuda'), target.to('cuda')

        optimizer.zero_grad()
        output = model(seq)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        train_loss+= loss.item() * seq.size(0)

    train_loss = train_loss/ len(train_loader.dataset)
    model.eval()
    val_loss = 0
    with torch.no_grad():
        # validation loop
        for seq, target in val_loader:
            seq, target = seq.to('cuda'), target.to('cuda')
    
            output = model(seq)
            loss = criterion(output, target)
    
            val_loss += loss.item() * seq.size(0)
    
        val_loss= val_loss/ len(val_loader.dataset)
    print(f"Epoch {epoch+1}/{num_epochs}, Training Loss: {train_loss:.6f}, Validation Loss: {val_loss:.6f}")

Epoch 1/20, Training Loss: 0.721166, Validation Loss: 0.560698
Epoch 2/20, Training Loss: 0.536394, Validation Loss: 0.488637
Epoch 3/20, Training Loss: 0.488050, Validation Loss: 0.465356
Epoch 4/20, Training Loss: 0.473480, Validation Loss: 0.457422
Epoch 5/20, Training Loss: 0.467725, Validation Loss: 0.455338
Epoch 6/20, Training Loss: 0.464458, Validation Loss: 0.453322
Epoch 7/20, Training Loss: 0.462299, Validation Loss: 0.451490


KeyboardInterrupt: 